In [10]:
import pandas as pd
from transformers import pipeline
from datasets import Dataset

In [9]:
import torch
print(torch.cuda.is_available())  # Should be True

True


In [2]:
#pretrained BERT models
# Sentiment model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Issue classifier
issue_model = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [3]:
#defining labels for issue
labels = [
    "Driver Behavior",
    "Delay / Late Arrival",
    "Pricing Issue",
    "Vehicle Condition",
    "App Issue",
    "Route Problem",
    "Cancellation Issue",
    "Payment Issue",
    "Other"
]

In [4]:
#function got prediction
def analyze_feedback(text):

    # Sentiment
    sent = sentiment_model(text)[0]

    # Issue classification
    issue = issue_model(text, labels)

    return {
        "text": text,
        "sentiment": sent['label'],
        "sentiment_score": round(sent['score'], 3),
        "top_issue": issue['labels'][0],
        "top_2_issues": issue['labels'][:2]
    }

In [5]:
#test
result = analyze_feedback("Driver was rude and arrived very late")

print(result)

{'text': 'Driver was rude and arrived very late', 'sentiment': 'NEGATIVE', 'sentiment_score': 0.997, 'top_issue': 'Delay / Late Arrival', 'top_2_issues': ['Delay / Late Arrival', 'Driver Behavior']}


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
#apply to dataset
df = pd.read_csv(r"/content/drive/MyDrive/rideflow_datasets.csv")

In [12]:
dataset = Dataset.from_pandas(df[['feedback_text']])

In [14]:
def get_sentiment(batch):
    results = sentiment_model(batch['feedback_text'], batch_size=32)
    return {
        "sentiment": [r['label'] for r in results],
        "sent_score": [r['score'] for r in results]
    }

dataset = dataset.map(get_sentiment, batched=True, batch_size=32)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [16]:
def get_issue(batch):
    results = issue_model(batch['feedback_text'], labels, batch_size=8)
    return {
        "issue": [r['labels'][0] for r in results],
        "issue_2": [r['labels'][1] for r in results]
    }

dataset = dataset.map(get_issue, batched=True, batch_size=8)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [17]:
df_result = dataset.to_pandas()
df_result.head()

,feedback_text,sentiment,sent_score,issue,issue_2
0,Driver was polite,POSITIVE,0.996525,Driver Behavior,Vehicle Condition
1,Driver cancelled suddenly,NEGATIVE,0.999438,Delay / Late Arrival,Cancellation Issue
2,Driver cancelled suddenly,NEGATIVE,0.999438,Delay / Late Arrival,Cancellation Issue
3,Good experience,POSITIVE,0.999860,Vehicle Condition,Driver Behavior
4,Vehicle was not clean,NEGATIVE,0.999702,Vehicle Condition,Other


In [30]:
df_result.to_csv(r"/content/drive/My Drive/feedback_analysis_output.csv", index=False)

In [32]:
#save model

from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Sentiment Model
sent_model_name = "distilbert-base-uncased-finetuned-sst-2-english"

sent_model = AutoModelForSequenceClassification.from_pretrained(sent_model_name)
sent_tokenizer = AutoTokenizer.from_pretrained(sent_model_name)

sent_model.save_pretrained("/content/drive/My Drive/models/sentiment_model")
sent_tokenizer.save_pretrained("/content/drive/My Drive/models/sentiment_model")


# Issue Model (Zero-shot)
issue_model_name = "facebook/bart-large-mnli"

issue_model = AutoModelForSequenceClassification.from_pretrained(issue_model_name)
issue_tokenizer = AutoTokenizer.from_pretrained(issue_model_name)

issue_model.save_pretrained("/content/drive/My Drive/models/issue_model")
issue_tokenizer.save_pretrained("/content/drive/My Drive/models/issue_model")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/My Drive/models/issue_model/tokenizer_config.json',
 '/content/drive/My Drive/models/issue_model/tokenizer.json')

In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "facebook/bart-large-mnli"

model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

model.save_pretrained(r"C:\Users\chaka\Preethu\My_Git_Repo\Final Project_Rideflow\saved_models\BERT_Feedback\models\issue_model")
tokenizer.save_pretrained(r"C:\Users\chaka\Preethu\My_Git_Repo\Final Project_Rideflow\saved_models\BERT_Feedback\models\issue_model")

c:\Users\chaka\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Writing model shards: 100%|██████████| 1/1 [00:09<00:00,  9.39s/it]


('C:\\Users\\chaka\\Preethu\\My_Git_Repo\\Final Project_Rideflow\\saved_models\\BERT_Feedback\\models\\issue_model\\tokenizer_config.json',
 'C:\\Users\\chaka\\Preethu\\My_Git_Repo\\Final Project_Rideflow\\saved_models\\BERT_Feedback\\models\\issue_model\\tokenizer.json')